# TMDB Movies Fetching

Pull extra movie metadata from the TMDB API and attach it to MovieLens IDs.

MovieLens only gives titles and genres. This notebook looks up each film on TMDB for overview, director, cast, keywords, runtime, language, and popularity — the richer content features we want for recommendations.

**Pipeline:** MovieLens `movies` + `links` → TMDB movie details → `data/tmdb_movies.csv`

## 1. Load MovieLens movies and TMDB IDs

Read `movies.csv` and `links.csv`, then join them on `movieId`.

`links` maps each MovieLens ID to a `tmdbId`. Without that ID we cannot call the TMDB API.

In [1]:
import pandas as pd

links = pd.read_csv("../data/links.csv")

movies = pd.read_csv("../data/movies.csv")

movies_with_ids = movies.merge(
    links,
    on="movieId",
    how="left"
)

print(movies_with_ids.shape)
movies_with_ids.head()


(87585, 5)


,movieId,title,genres,imdbId,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844.0
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357.0
4,5,Father of the Bride Part II (1995),Comedy,113041,11862.0


## 2. Load the TMDB API token

Read `TMDB_ACCESS_TOKEN` from `.env` and build the request headers.

Keep the token in `.env` so it is not committed.

In [2]:
import os
import requests
from dotenv import load_dotenv

load_dotenv("../.env")

TMDB_ACCESS_TOKEN = os.getenv("TMDB_ACCESS_TOKEN")

headers = {
    "Authorization": f"Bearer {TMDB_ACCESS_TOKEN}",
    "accept": "application/json"
}

## 3. Fetch one movie from TMDB

`fetch_tmdb_movie(tmdb_id)` calls `GET /movie/{id}` and also requests `credits` and `keywords` in the same response.

Returns JSON on success, or `None` if the request fails.

In [3]:
def fetch_tmdb_movie(tmdb_id):
    url = f"https://api.themoviedb.org/3/movie/{int(tmdb_id)}"

    params = {
        "append_to_response": "credits,keywords"
    }

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=20
    )

    if response.status_code == 200:
        return response.json()

    return None

## 4. Flatten the TMDB JSON into a row

`parse_tmdb_movie` turns the nested API payload into one flat dict for CSV.

It keeps overview, genres, runtime, language, country, popularity, votes, budget, revenue, director(s), top 10 cast, keywords, and production companies. List fields are joined with `|`.

In [4]:
def parse_tmdb_movie(data, movie_id=None):
    if data is None:
        return None

    # Genres
    genres = [g["name"] for g in data.get("genres", [])]

    # Director
    crew = data.get("credits", {}).get("crew", [])
    directors = [
        person["name"]
        for person in crew
        if person.get("job") == "Director"
    ]

    # Top 10 cast
    cast = data.get("credits", {}).get("cast", [])
    top_cast = [
        person["name"]
        for person in cast[:10]
    ]

    # Keywords
    keywords = [
        k["name"]
        for k in data.get("keywords", {}).get("keywords", [])
    ]

    # Production companies
    companies = [
        company["name"]
        for company in data.get("production_companies", [])
    ]

    return {
        "movieId": movie_id,
        "tmdbId": data.get("id"),
        "title": data.get("title"),
        "overview": data.get("overview"),
        "genres": "|".join(genres),
        "runtime": data.get("runtime"),
        "original_language": data.get("original_language"),
        "origin_country": "|".join(data.get("origin_country", [])),
        "popularity": data.get("popularity"),
        "vote_average": data.get("vote_average"),
        "vote_count": data.get("vote_count"),
        "budget": data.get("budget"),
        "revenue": data.get("revenue"),
        "director": "|".join(directors),
        "top_cast": "|".join(top_cast),
        "keywords": "|".join(keywords),
        "production_companies": "|".join(companies)
    }

## 5. Batch fetch, checkpoint, and save (Complete)
`fetch_and_save_tmdb` walks the movie list and writes results to `data/tmdb_movies.csv`.



In [5]:
"""
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd

OUTPUT_FILE = Path("../data/tmdb_movies.csv")


def fetch_one_movie(row):
    movie_id = int(row["movieId"])
    tmdb_id = int(row["tmdbId"])

    try:
        data = fetch_tmdb_movie(tmdb_id)

        if data is not None:
            return parse_tmdb_movie(data, movie_id=movie_id)

    except Exception as e:
        print(f"Error: movieId={movie_id}, tmdbId={tmdb_id}: {e}")

    return None


def fetch_and_save_tmdb(
    movies_df,
    max_workers=10,
    batch_size=1000
):
    # Read existing checkpoint
    if OUTPUT_FILE.exists():
        existing_df = pd.read_csv(OUTPUT_FILE)

        completed_ids = set(
            existing_df["movieId"]
            .dropna()
            .astype(int)
        )

        print(f"Found {len(completed_ids)} previously fetched movies.")

    else:
        existing_df = pd.DataFrame()
        completed_ids = set()

    # Only movies that still need fetching
    to_fetch = movies_df[
        movies_df["tmdbId"].notna()
        & ~movies_df["movieId"].isin(completed_ids)
    ].copy()

    total_remaining = len(to_fetch)

    print(f"Movies remaining: {total_remaining}")

    # Process automatically in batches
    for start in range(0, total_remaining, batch_size):

        batch = to_fetch.iloc[start:start + batch_size]

        batch_results = []

        print(
            f"\nBatch {start // batch_size + 1}: "
            f"{start + 1}-{min(start + batch_size, total_remaining)}"
        )

        with ThreadPoolExecutor(max_workers=max_workers) as executor:

            futures = [
                executor.submit(fetch_one_movie, row)
                for _, row in batch.iterrows()
            ]

            for future in tqdm(
                as_completed(futures),
                total=len(futures)
            ):
                result = future.result()

                if result is not None:
                    batch_results.append(result)

        # Save this batch
        if batch_results:

            batch_df = pd.DataFrame(batch_results)

            if OUTPUT_FILE.exists():
                batch_df.to_csv(
                    OUTPUT_FILE,
                    mode="a",
                    header=False,
                    index=False
                )
            else:
                batch_df.to_csv(
                    OUTPUT_FILE,
                    index=False
                )

        print(f"Saved batch: {len(batch_results)} movies")

    print("\nFinished fetching all available TMDB movies.")
"""

'\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom pathlib import Path\nfrom tqdm.auto import tqdm\nimport pandas as pd\n\nOUTPUT_FILE = Path("../data/tmdb_movies.csv")\n\n\ndef fetch_one_movie(row):\n    movie_id = int(row["movieId"])\n    tmdb_id = int(row["tmdbId"])\n\n    try:\n        data = fetch_tmdb_movie(tmdb_id)\n\n        if data is not None:\n            return parse_tmdb_movie(data, movie_id=movie_id)\n\n    except Exception as e:\n        print(f"Error: movieId={movie_id}, tmdbId={tmdb_id}: {e}")\n\n    return None\n\n\ndef fetch_and_save_tmdb(\n    movies_df,\n    max_workers=10,\n    batch_size=1000\n):\n    # Read existing checkpoint\n    if OUTPUT_FILE.exists():\n        existing_df = pd.read_csv(OUTPUT_FILE)\n\n        completed_ids = set(\n            existing_df["movieId"]\n            .dropna()\n            .astype(int)\n        )\n\n        print(f"Found {len(completed_ids)} previously fetched movies.")\n\n    else:\n        existing_df = pd

In [6]:
"""
fetch_and_save_tmdb(
    movies_with_ids,
    max_workers=10,
    batch_size=1000
)
"""

'\nfetch_and_save_tmdb(\n    movies_with_ids,\n    max_workers=10,\n    batch_size=1000\n)\n'

87,461 movies had a TMDB ID, and approximately 86,250 were successfully enriched, giving ~98.6% coverage.

# TMDB Data Preprocessing

### Load and Inspect TMDB Dataset

In [7]:
tmdb_df = pd.read_csv("../data/tmdb_movies.csv")

print("Shape:", tmdb_df.shape)
print("Duplicate movieIds:", tmdb_df["movieId"].duplicated().sum())
print("Duplicate tmdbIds:", tmdb_df["tmdbId"].duplicated().sum())

tmdb_df.info()

Shape: (86250, 17)
Duplicate movieIds: 0
Duplicate tmdbIds: 35
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86250 entries, 0 to 86249
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   movieId               86250 non-null  int64  
 1   tmdbId                86250 non-null  int64  
 2   title                 86250 non-null  object 
 3   overview              86013 non-null  object 
 4   genres                85597 non-null  object 
 5   runtime               86250 non-null  int64  
 6   original_language     86250 non-null  object 
 7   origin_country        86126 non-null  object 
 8   popularity            86250 non-null  float64
 9   vote_average          86250 non-null  float64
 10  vote_count            86250 non-null  int64  
 11  budget                86250 non-null  int64  
 12  revenue               86250 non-null  int64  
 13  director              85770 non-null  object 
 14  top_cas

### Check Missing Values

In [8]:
missing = pd.DataFrame({
    "missing_count": tmdb_df.isna().sum(),
    "missing_percent": tmdb_df.isna().mean() * 100
})

missing.sort_values("missing_percent", ascending=False)

,missing_count,missing_percent
keywords,23418,27.151304
production_companies,9989,11.581449
top_cast,3076,3.566377
genres,653,0.757101
director,480,0.556522
overview,237,0.274783
origin_country,124,0.143768
vote_count,0,0.000000
revenue,0,0.000000
budget,0,0.000000


### Check Zero Values

In [9]:
for col in [
    "runtime",
    "popularity",
    "vote_average",
    "vote_count",
    "budget",
    "revenue"
]:
    print(
        col,
        "zero:",
        (tmdb_df[col] == 0).sum(),
        "min:",
        tmdb_df[col].min(),
        "max:",
        tmdb_df[col].max()
    )

runtime zero: 649 min: 0 max: 12480
popularity zero: 21 min: 0.0 max: 135.4648
vote_average zero: 951 min: 0.0 max: 10.0
vote_count zero: 946 min: 0 max: 41141
budget zero: 69822 min: 0 max: 700000000
revenue zero: 70037 min: 0 max: 2923706026


Since ~81% of the values from features "budget" and "revenue" are 0, they will not be used for future modelling, as they will not provide any useful information and will introduce noise.

### Checking Features

In [17]:
import pandas as pd
import numpy as np

# ============================================================
# 1. BASIC DATASET INFORMATION
# ============================================================

print("=" * 70)
print("1. BASIC DATASET INFORMATION")
print("=" * 70)

print("Shape:", tmdb_df.shape)
print("Duplicate movieId:", tmdb_df["movieId"].duplicated().sum())
print("Duplicate tmdbId:", tmdb_df["tmdbId"].duplicated().sum())


# ============================================================
# 2. MISSING VALUES
# ============================================================

print("\n" + "=" * 70)
print("2. MISSING VALUES")
print("=" * 70)

missing_summary = pd.DataFrame({
    "missing_count": tmdb_df.isna().sum(),
    "missing_percent": (tmdb_df.isna().mean() * 100).round(2)
})

print(
    missing_summary
    .sort_values("missing_percent", ascending=False)
    .to_string()
)


# ============================================================
# 3. NUMERICAL FEATURES
# ============================================================

print("\n" + "=" * 70)
print("3. NUMERICAL FEATURES")
print("=" * 70)

numerical_cols = [
    "runtime",
    "popularity",
    "vote_average",
    "vote_count",
    "budget",
    "revenue"
]

numerical_summary = tmdb_df[numerical_cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T

numerical_summary["zero_count"] = (
    tmdb_df[numerical_cols] == 0
).sum()

numerical_summary["zero_percent"] = (
    (tmdb_df[numerical_cols] == 0).mean() * 100
).round(2)

print(numerical_summary.to_string())


# ============================================================
# 4. CATEGORICAL / TEXT FEATURES
# ============================================================

print("\n" + "=" * 70)
print("4. CATEGORICAL / TEXT FEATURES")
print("=" * 70)

categorical_cols = [
    "title",
    "overview",
    "genres",
    "original_language",
    "origin_country",
    "director",
    "top_cast",
    "keywords",
    "production_companies"
]

categorical_summary = pd.DataFrame({
    "unique_values": [
        tmdb_df[col].nunique()
        for col in categorical_cols
    ],
    "missing_count": [
        tmdb_df[col].isna().sum()
        for col in categorical_cols
    ],
    "missing_percent": [
        round(tmdb_df[col].isna().mean() * 100, 2)
        for col in categorical_cols
    ]
}, index=categorical_cols)

print(categorical_summary.to_string())


# ============================================================
# 5. MOST COMMON VALUES
# ============================================================

print("\n" + "=" * 70)
print("5. MOST COMMON CATEGORICAL VALUES")
print("=" * 70)

for col in [
    "original_language",
    "origin_country",
    "genres"
]:
    print(f"\n--- {col} ---")
    print(tmdb_df[col].value_counts(dropna=False).head(10))


# ============================================================
# 6. TEXT LENGTHS
# ============================================================

print("\n" + "=" * 70)
print("6. TEXT FEATURE LENGTHS")
print("=" * 70)

for col in [
    "overview",
    "director",
    "top_cast",
    "keywords",
    "production_companies"
]:
    lengths = tmdb_df[col].fillna("").str.len()

    print(
        f"{col:25} "
        f"median={lengths.median():.0f}, "
        f"95%={lengths.quantile(0.95):.0f}, "
        f"max={lengths.max()}"
    )


# ============================================================
# 7. FINAL CHECK
# ============================================================

print("\n" + "=" * 70)
print("7. DATA TYPES")
print("=" * 70)

print(tmdb_df.dtypes)

1. BASIC DATASET INFORMATION
Shape: (86250, 17)
Duplicate movieId: 0
Duplicate tmdbId: 35

2. MISSING VALUES
                      missing_count  missing_percent
movieId                           0              0.0
vote_average                      0              0.0
keywords                          0              0.0
top_cast                          0              0.0
director                          0              0.0
revenue                           0              0.0
budget                            0              0.0
vote_count                        0              0.0
popularity                        0              0.0
tmdbId                            0              0.0
origin_country                    0              0.0
original_language                 0              0.0
runtime                           0              0.0
genres                            0              0.0
overview                          0              0.0
title                             0        

Removing features `budget`, `revenue`, `popularity`, `vote_average`, `vote_count`

In [18]:
remove_features = [
    "budget",
    "revenue",
    "popularity",
    "vote_average",
    "vote_count"
]

### Handling Missing Values

In [16]:
text_features = [
    "overview",
    "genres",
    "director",
    "top_cast",
    "keywords",
    "production_companies",
    "origin_country"
]

tmdb_df[text_features] = tmdb_df[text_features].fillna("Unknown")

print(tmdb_df[text_features].isna().sum())

overview                0
genres                  0
director                0
top_cast                0
keywords                0
production_companies    0
origin_country          0
dtype: int64
